# ailm_inspectable on Colab

Runs the instrumented model service on Colab GPU and exposes it through a
cloudflared quick tunnel. Any OpenAI-compatible client pointed at the
printed URL gets chat completions plus an internals receipt (per-token
top-k, logit-lens trajectory, attention stats, residual norms).

**Steps:** Runtime -> Change runtime type -> GPU (T4 is enough for the
default model). Run cells top to bottom. Cell 4 fetches
`ailm_inspectable.py` from this repo (falls back to an embedded copy if
unreachable). The last cell is a keep-alive loop -- leave it running;
the VM dies when the notebook closes or idles out.

Point a client at the printed URL:

```bash
export OPENAI_BASE_URL=<printed URL>/v1
export OPENAI_API_KEY=<printed key>
```


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q torch transformers accelerate fastapi "uvicorn[standard]"
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/bin/cloudflared
!chmod +x /usr/bin/cloudflared
print('deps ok')

In [ ]:
import secrets, subprocess, os

# model picks by accelerator memory: <20GB->1.5B  >=20GB->7B  >=60GB->14B (tight)
def pick_model():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.total',
             '--format=csv,noheader,nounits'], text=True)
        mem_gb = float(out.strip().splitlines()[0]) / 1024
    except Exception:
        mem_gb = 0
    if mem_gb >= 60:
        return "Qwen/Qwen2.5-14B-Instruct"
    if mem_gb >= 20:
        return "Qwen/Qwen2.5-7B-Instruct"
    return "Qwen/Qwen2.5-1.5B-Instruct"

MODEL = pick_model()  # ungated; for gated models set HF_TOKEN below
API_KEY = secrets.token_urlsafe(24)

os.environ['AILM_API_KEY'] = API_KEY
# generation bound per call; 2048 leaves room for multi-section
os.environ['AILM_MAX_NEW_TOKENS'] = '2048'
# os.environ['HF_TOKEN'] = 'hf_...'   # needed only for gated models (gemma, llama)
print('GPU ->', MODEL)
print('API_KEY (goes to the Mac):', API_KEY)


In [ ]:
import base64, os, urllib.request

RAW_URL = 'https://raw.githubusercontent.com/barlowa124/ailm-inspectable/main/ailm_inspectable.py'

# embedded copy of the pinned revision - used when RAW_URL is unreachable
B64 = '...'

def fetch_remote():
    req = urllib.request.Request(RAW_URL)
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')  # Colab Secret, if the URL is private
        if tok:
            req.add_header('Authorization', 'token ' + tok)
    except Exception:
        pass
    return urllib.request.urlopen(req, timeout=60).read()

try:
    data = fetch_remote()
    open('/content/ailm_inspectable.py', 'wb').write(data)
    print('fetched from', RAW_URL, '(%d bytes)' % len(data))
except Exception as e:
    open('/content/ailm_inspectable.py', 'wb').write(base64.b64decode(B64))
    print('fetch failed (%s); wrote embedded copy' % type(e).__name__)
assert os.path.exists('/content/ailm_inspectable.py')


In [ ]:
import re, subprocess, threading

svc_log = open('/content/inspectable.log', 'w')
svc = subprocess.Popen(
    ['python3', 'ailm_inspectable.py', '--model', MODEL, '--port', '8000'],
    stdout=svc_log, stderr=subprocess.STDOUT)
print('service pid', svc.pid, '- first start downloads the model')

tun = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000',
     '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

TUNNEL_URL = None
for line in tun.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        TUNNEL_URL = m.group(0)
        break
assert TUNNEL_URL, 'cloudflared never printed a tunnel URL'
threading.Thread(target=lambda: [l for l in tun.stdout],
                 daemon=True).start()  # drain pipe

print('TUNNEL_URL =', TUNNEL_URL)
print()
print('on the Mac:')
print(f'  export AILM_REMOTE_URL={TUNNEL_URL}/v1')
print(f'  export AILM_REMOTE_API_KEY={API_KEY}')

In [ ]:
import json, time, urllib.request

for _ in range(120):
    try:
        h = json.loads(urllib.request.urlopen(
            'http://127.0.0.1:8000/health', timeout=5).read())
        break
    except Exception:
        time.sleep(5)
else:
    print(open('/content/inspectable.log').read()[-3000:])
    raise SystemExit('service never came up - check the log above')
print('healthy:', h)

req = urllib.request.Request(
    TUNNEL_URL + '/v1/chat/completions',
    data=json.dumps({'model': MODEL, 'max_tokens': 8, 'messages': [
        {'role': 'user', 'content': 'Reply with just the word ok'}]}).encode(),
    headers={'Content-Type': 'application/json',
             'Authorization': f'Bearer {API_KEY}'})
out = json.loads(urllib.request.urlopen(req, timeout=300).read())
print('reply through tunnel:', out['choices'][0]['message']['content'])
print('internals keys:', sorted(out['ailm']['internals']))

In [ ]:
import json, time, urllib.request

print('keep-alive loop running; stop this cell to shut the service down')
while True:
    time.sleep(300)
    try:
        h = json.loads(urllib.request.urlopen(
            'http://127.0.0.1:8000/health', timeout=10).read())
        print(time.strftime('%H:%M'), h['model'], 'on', h['device'])
    except Exception as e:
        print('health check failed:', e)
        break